
# Feature Engineering: Temporal + Base Dataset Assembly

## Objective

In this notebook, I combine all yearly datasets and create temporal features such as lag variables and rolling statistics.

This forms the base dataset that will later be combined with vegetation and human interaction features.

In [1]:

import pandas as pd
import numpy as np



## Step 1: Combine Data Across All Years

Here I load individual yearly datasets (2018–2023) and merge them into a single dataset.

This ensures that the model will be trained on multiple years of data rather than a single year.



## Dataset Overview

After combining all years, I check:

- Total number of rows
- Structure of the dataset

This confirms that the dataset includes all spatial and temporal observations across the study period.


In [2]:

dfs = [pd.read_parquet(f"df_{y}.parquet") for y in range(2018, 2024)]
df = pd.concat(dfs, ignore_index=True)

print(df.shape)
df.head()


(35546784, 8)


,day,lat,lon,fire,pr,tmmx,tmmn,year
0,2018-01-01,40.983333,-109.058333,0,0.0,1.450012,-12.350006,2018
1,2018-01-01,40.983333,-109.016667,0,0.0,2.050018,-13.149994,2018
2,2018-01-01,40.983333,-108.975000,0,0.0,1.050018,-13.250000,2018
3,2018-01-01,40.983333,-108.933333,0,0.0,0.649994,-13.149994,2018
4,2018-01-01,40.983333,-108.891667,0,0.0,0.950012,-12.449982,2018



## Step 2: Sort Data by Location and Time

The dataset is sorted by:

- Latitude
- Longitude
- Day

This is important because lag and rolling operations require the time series to be in the correct order for each location.


In [3]:
df = df.sort_values(["lat", "lon", "day"]).reset_index(drop=True)


## Step 3: Create Day-of-Year Feature

I create a "day of year" variable to represent seasonality.

This helps capture long-term seasonal trends in wildfire risk.


In [4]:
df["doy"] = df["day"].dt.dayofyear


## Step 4: Create Lag Features

I compute lagged precipitation features:

- 1-day lag
- 3-day lag
- 7-day lag

These features capture recent weather conditions leading up to a given day.


In [5]:

df["pr_lag1"] = df.groupby(["lat", "lon"])["pr"].shift(1)
df["pr_lag3"] = df.groupby(["lat", "lon"])["pr"].shift(3)
df["pr_lag7"] = df.groupby(["lat", "lon"])["pr"].shift(7)



## Step 5: Create Rolling Aggregates

I compute rolling precipitation sums over:

- 7 days
- 14 days

These features represent accumulated moisture conditions, which strongly influence wildfire risk.


In [6]:

df["pr_roll7"]  = df.groupby(["lat", "lon"])["pr"].rolling(7).sum().reset_index(level=[0,1], drop=True)
df["pr_roll14"] = df.groupby(["lat", "lon"])["pr"].rolling(14).sum().reset_index(level=[0,1], drop=True)



## Step 6: Handle Missing Values

Lag and rolling calculations introduce missing values at the beginning of each time series.

I remove these rows to ensure a clean dataset for modeling.



## Dataset Size After Processing

I check the size of the dataset after removing missing values.

This confirms how much data is retained for modeling.


In [7]:

df = df.dropna().reset_index(drop=True)
print(df.shape)


(35335872, 14)


In [8]:

df.to_parquet("df_features_2018_2023.parquet")
print("✅ Saved df_features_2018_2023.parquet")


✅ Saved df_features_2018_2023.parquet



## Step 7: Save Processed Dataset

The final dataset is saved as a Parquet file:

## Conclusion

In this notebook, I created the base feature dataset by:

- Combining multiple years of data (2018–2023)
- Adding temporal features such as lags and rolling summaries
- Cleaning and preparing the dataset for modeling

This dataset forms the foundation for the next step, where additional spatial features (vegetation and road distance) will be incorporated.
